# HW02 - Полный цикл Разведочного анализа данных (EDA)

Dataset [Video game sale](https://www.kaggle.com/datasets/gregorut/videogamesales) 

Дата сет о продажах игр от разных компаний

## 2. A) Быстрый обзор данных (Pandas)
1. df.head(), df.tail(), df.shape.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce
import plotly.express as px

In [ ]:
df = pd.read_csv("vgsales.csv")

print("Первые 5 строк:")
display(df.head())
print("Последние 5 строк:")
display(df.tail())
print(f"Shape: {df.shape}")

display(df.Publisher.unique())

2. df.info() и df.describe() (отдельно покажите .describe(include="object") для строк).

In [ ]:
print(f"Info:\n{df.info()}")
print("Статистика:")
display(df.describe())
display(df.describe(include=['object', 'string']))

3. Проверки качества: isnull().sum(), поиск дубликатов, проверка адекватности типов данных.

In [ ]:
dupl = df.duplicated().sum()
print(f"Количество дубликатов: {dupl}")
display(df.duplicated())

print(f"Используемые типы:\n {df.dtypes}")
print(f"\nПроверка качества:\n{df.isnull().sum()}")

## Б) Пропуски и очистка
1. Покажите стратегии dropna() и fillna(). Заполняйте средним, медианой, константой — и обязательно обоснуйте выбор текстовым комментарием.



In [ ]:
new_df = df.dropna()
print(f"Строк до удаления {len(df)}")
print(f"Строк после удаления {len(new_df)}")

""" В годах считаем медиан, чтобы игнорировать резкие выбросы. 
    Также у нас год не может быть дробным, что можем быть с mean"""

df["Year"] = df["Year"].fillna(df["Year"].median())

""" В создателях выюираем наиболее встречающегося,
    тк скорее всего он и будет создателем.
    Также данные не являются числовыми,
    поэтому использовать mean и meadian не можем"""
df["Publisher"] = df["Publisher"].fillna(df["Publisher"].mode()[0])

print(f"Статистика после заполнени:\n{df.isnull().sum()}")

2. Отдельно покажите, как считаете моду (mode) и заполняете ею пропуски в категориальных колонках.

In [ ]:
#Каждый раз берем mode()[0], тк mode представляется в виде series
print("Наиболее встречающееся для name: ", df["Name"].mode()[0])
print("Для Platform: ", df["Platform"].mode()[0])
print("Для Genre: ", df["Genre"].mode()[0])
print("Для Publisher: ", df["Publisher"].mode()[0])

## C) Расширенная статистика
1. Для числовых колонок выведите: min, max, mean, median, mode.



In [ ]:
num_df = df.select_dtypes(include = ["number"])

res = num_df.describe()
res.loc["mode"] = num_df.mode().iloc[0]
res.loc["median"] = num_df.median()
display(res.loc[["min", "max", "mean", "median", "mode"]])

2. Посчитайте percentile/quantile (например, 5, 25, 50, 75, 95 перцентили).

In [ ]:
res.loc["5%"] = num_df.quantile(0.05)
res.loc["95%"] = num_df.quantile(0.95)
display(res.loc[["5%", "25%", "50%", "75%", "95%"]])

3. 🧠 Самостоятельно изучите и посчитайте: дисперсию (variance), асимметрию (skewness) и эксцесс (kurtosis). Попробуйте объяснить, что они значат для ваших данных.

In [ ]:
print("Дисперсия:\n", num_df.var())
"""Дисперсия помогает посмотреть нащ разброс для среднего,
   например в годах разница между поздним и ранним 33 года"""
print("Ассиметрия:\n", num_df.skew())
"""Показывает куда завалены наши данные,
   у нас ассиметрия завалена вправо, а это значит
   что большинство игр было неинтересно"""
print("Эксцесс:\n", num_df.kurt())
"""Показывает равномерные ли у нас данные
   или есть выбросы"""

## D) Фичи: Энкодинг и Инжиниринг (Feature Engineering)
1. Сделайте кодирование категорий: OneHotEncoder (или pd.get_dummies) — обязательно.




In [ ]:
print(df["Genre"].nunique())
df_onehot = pd.get_dummies(df, columns = ["Genre"])
display(df_onehot)

In [ ]:
df_onehot2 = pd.get_dummies(df, columns = ["Genre", "Platform"])
print("Размер оригинального датафрейма", df.shape)
print("Размер после энкодинга: ", df_onehot2.shape)
display(df_onehot2)

2. Если уместно: Label Encoding или Target Encoding. Покажите датафрейм до/после.

In [ ]:
print("Датафрейм до")
display(df.head())
print("После")

le = LabelEncoder()

copy_df = df.copy()
copy_df["Genre_Encoder"] = le.fit_transform(copy_df["Genre"])
display(copy_df)

In [ ]:
print("Датафрейм до")
display(df.head())

print("После")
df_target = df.groupby("Publisher")["Global_Sales"].mean()

copy_df2 = df.copy()
copy_df2["Publisher_Target"] = copy_df2["Publisher"].map(df_target)
display(copy_df2)

3. 🧠 Самостоятельно изучите: попробуйте применить Feature Hashing (Hashing Encoder) для признаков с большим числом уникальных значений.

In [ ]:
hasher = ce.HashingEncoder(cols=['Publisher'], n_components=10)

df_hashed = hasher.fit_transform(df)
display(df_hashed)

4. 🧠 Генерация новых фич: создайте 2-3 новых признака. Например, перемножьте/поделите две существующие колонки, вытащите месяц из даты или сгруппируйте редкие категории в одну "Other".

In [ ]:
western_df = df.copy()

western_df["Western_Sales"] = df["NA_Sales"] + df["EU_Sales"]
display(western_df[["Western_Sales", "NA_Sales", "EU_Sales"]])

In [ ]:
nowestern_df = df.copy()

nowestern_df["NOWESTERN_Sales"] = df["JP_Sales"] + df["Other_Sales"]
display(nowestern_df[["NOWESTERN_Sales", "JP_Sales", "Other_Sales"]])

## E) Визуализация (Самая красивая часть!)
Используем ВСЕ ТРИ библиотеки (matplotlib, seaborn, plotly). Не для галочки, а со смыслом!
1. Распределения: гистограммы / KDE.





In [ ]:
plt.figure(figsize=(6,8))
sns.histplot(data = df["Year"], kde = True, bins = 30)
plt.xlabel("Год")
plt.ylabel("Количество игр")
plt.title("Продажа игр по годам")
plt.show()


2. Scatter plot — ищем зависимости между числами.

In [ ]:
plt.figure(figsize = (6,8))
sns.scatterplot(data = df, x = "Year", y = "Global_Sales", hue = "Genre")
plt.title("Продажа игр по годам на scatterplot")
plt.show()

In [ ]:
plt.figure(figsize = (6,6))
sns.scatterplot(data = df,x="EU_Sales", y = "NA_Sales",hue = "Genre", alpha = 0.6)
plt.xlim(0,29)
plt.ylim(0,41)
plt.title("Зависимость между регионами")
plt.show()

3. Box plot (ящик с усами) — ищем аномалии/выбросы.

In [ ]:
new_df = df[df["Publisher"].isin(["Nintendo", "Microsoft Game Studios"])]
sum_df = new_df.groupby("Publisher")["Global_Sales"].sum().index

plt.figure(figsize = (6,8))
sns.boxplot(data = new_df, y = "Global_Sales", x = "Publisher",showfliers=False, order = sum_df)
plt.ylabel("Продано млн")
plt.show()

In [ ]:
plt.figure(figsize = (6,8))
sns.boxplot(data = df, y = "Genre", x = "Global_Sales", showfliers = False, notch = False)
plt.title("Ящик с усами")
plt.show()

4. Bar/count plot для категорий.

In [ ]:
plt.figure(figsize = (12,8))
sns.barplot(data = df, x = "Genre", y = "Global_Sales", errorbar = None)
plt.ylabel("Глобальные продажи")
plt.title("Продажи по жанрам")
plt.show()


In [ ]:
regions = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
region_sums = df[regions].sum()

plt.figure(figsize = (6,8))
sns.barplot(x=region_sums.index, y=region_sums.values)
plt.title("Общие продажи по регионам")
plt.xlabel("Регионы")
plt.show()

In [ ]:
#new_df из 3 пункта

plt.figure(figsize = (10,8))
sns.countplot(data = new_df, x = "Publisher", hue = "Genre")
plt.legend(title ="Жанр")
plt.ylabel("Продажи")
plt.show()

5. Heatmap корреляций (напишите короткий вывод, какие колонки сильно коррелируют).
(По желанию: pairplot, violinplot, интерактивный график Plotly с параметром hover_data).

In [ ]:
corr_matrix = df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']].corr()

plt.figure(figsize = (8,8))
sns.heatmap(corr_matrix, annot =True)
plt.show()

In [ ]:
plt.figure(figsize = (12,8))
sns.violinplot(df, x = "Genre", y = "Global_Sales")
plt.ylim(0,4)
plt.show()

In [ ]:
plt.figure(figsize = (8,8))
sns.pairplot(df[["NA_Sales", "EU_Sales", "JP_Sales", "Genre"]].sample(500), hue = "Genre")
plt.show()

In [ ]:
fig = px.treemap(df, path=["Genre", "Platform"], values="Global_Sales", title="Распределение продаж по жанрам и платформам")
fig.show()

## F) Итоговые выводы (Markdown)
   В конце ноутбука напишите:
1. 7–12 буллетов «Что я понял про датасет».

    Датасет - набор данных, которые хранятся в виде таблицы.

    В датасете столбцы выступают признаками, а строки обектами.

    С помощью датасета можно найти зависимости, построить графики.

    В моем датасете представлены данные об играх разных годов.

    Из датасета понятно, что Североамериканский рынок - лидер по продажам.

    Самый популярный жанр - Platform.
    
    Продажи в Америке и Европе сильно связаны (это видно по heatmap)


2. 3–5 гипотез/наблюдений (про влияние категорий на таргет, природу выбросов и т.д.).

Большая популярность у игр появилась после 2000 года, это видно по гистограмме

В моем датасете очень много выбросов, пришлось в boxplotах отключать их, один из выбросов есть на scetter plotе

При кодировании выбрал Labelencoding для жанров, тк их немного(также для жанров использовал OneHot, чтобы посмотреть что выйдет)

Из heatmap видно, что у Японии самая низкая корреляция

3. Что бы вы сделали дальше (какую модель взяли бы, что предсказывали).

Хочу взять про определнии типа транзакции, еще можно было бы предсказать курс биткойна

4. Маленький блок: «Какие подсказки брал у AI и что в итоге проверял/дописывал руками».

У ИИ спрашивал про параметры в графиках, про некотрые группировки данных и про encoding и engineering